<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase26_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICHI 2027 — Phase 26
## Synthetic FHIR R4 Evidence Representation + Contract Validation
**Drive → Drive notebook**

### Purpose
This phase starts only after Phase 25 has established that:

- the exact original generic TFL producer is **not portable to MRI as-is**;
- Arm C must therefore remain recorded as non-executable on MRI rather than being silently rewritten;
- the imaging-specific contract is currently **synthetic prototype only**.

Phase 26 creates a reproducible **synthetic FHIR R4 evidence layer** for longitudinal neuroimaging and checks whether the preservation dimensions defined in the blueprint can be represented and reconstructed.

### This notebook does **not** claim
- pyOMOP execution;
- OMOP or MI-CDM loading;
- Athena concept approval;
- validation against real ADNI records;
- official HL7 profile conformance;
- DICOM UID attestation;
- a completed four-arm comparison.

The notebook exports only synthetic/public-safe artifacts.


In [1]:

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

# Imports and Drive discovery
from pathlib import Path
import json, csv, hashlib, copy
from datetime import datetime, timezone
import pandas as pd

MYDRIVE = Path("/content/drive/MyDrive")

phase25_candidates = sorted(
    MYDRIVE.rglob("ICHI2027_Phase25_SHAREABLE*.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not phase25_candidates:
    raise FileNotFoundError(
        "Could not find ICHI2027_Phase25_SHAREABLE*.json anywhere under MyDrive."
    )

PHASE25_JSON = phase25_candidates[0]
WORKDIR = PHASE25_JSON.parent

summary_candidates = sorted(
    WORKDIR.glob("ICHI2027_Phase25_SYNTHETIC_FIXTURE_SUMMARY*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not summary_candidates:
    raise FileNotFoundError(
        f"Phase 25 JSON found at {PHASE25_JSON}, but no matching Phase 25 summary CSV was found in {WORKDIR}."
    )

PHASE25_CSV = summary_candidates[0]

print("Phase 25 JSON :", PHASE25_JSON)
print("Phase 25 CSV  :", PHASE25_CSV)
print("Phase 26 output folder:", WORKDIR)


Phase 25 JSON : /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase25_SHAREABLE.json
Phase 25 CSV  : /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase25_SYNTHETIC_FIXTURE_SUMMARY.csv
Phase 26 output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


In [3]:

# Verify Phase 25 gate before continuing
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with open(PHASE25_JSON, "r", encoding="utf-8") as f:
    phase25 = json.load(f)

phase25_summary = pd.read_csv(PHASE25_CSV)

required_phase = "25_generic_nonportability_disposition_synthetic_contract_gate"
assert phase25.get("phase") == required_phase, f"Unexpected Phase 25 phase label: {phase25.get('phase')}"
assert phase25.get("gates", {}).get("phase24_json_verified") is True
assert phase25.get("gates", {}).get("exact_original_generic_C_portable_to_MRI") is False
assert phase25.get("gates", {}).get("exact_original_generic_C_MRI_executed") is False
assert phase25.get("gates", {}).get("synthetic_prototype_contract_checks_passed") is True
assert phase25.get("synthetic_total_fixtures") == 10
assert phase25.get("synthetic_fixtures_passing_declared_expectations") == 10

print("Phase 25 gate verified.")
print("Phase 25 SHA256:", sha256_file(PHASE25_JSON))


Phase 25 gate verified.
Phase 25 SHA256: 42f73f55c3671c02dec1c00d09016ff30371f9c1343fac268b262e6353a4861d



## Phase 26 synthetic evidence contract

Each clean longitudinal evidence object carries these dimensions:

1. synthetic patient reference;
2. imaging study/acquisition reference;
3. time point;
4. anatomy;
5. laterality;
6. biomarker name;
7. numeric value;
8. UCUM unit;
9. extraction software name/version;
10. TRACE state and rule version;
11. provenance chain linking the derived measurement to its ImagingStudy and Device.

FHIR R4 resources used here:

- `Patient`
- `ImagingStudy`
- `Device`
- `Observation`
- `Provenance`

These are synthetic structures for research-method development. This notebook performs a **project-specific structural contract check**, not formal HL7 validator certification.


In [4]:

# Constants for the synthetic public-safe fixture set
SYSTEM = "https://example.org/ichi2027/synthetic"
UCUM = "http://unitsofmeasure.org"

ANATOMY_CODES = {
    "hippocampus": "SYN-HIP",
    "entorhinal": "SYN-ENT",
}

TRACE_STATES = ["PASS", "CAUTION", "REVIEW_REQUIRED"]

def ref(resource_type, rid):
    return {"reference": f"{resource_type}/{rid}"}

def make_patient(pid):
    return {
        "resourceType": "Patient",
        "id": pid,
        "identifier": [{"system": SYSTEM + "/patient", "value": pid}],
    }

def make_device(did, software_name="SynthSegLike", version="1.0"):
    d = {
        "resourceType": "Device",
        "id": did,
        "identifier": [{"system": SYSTEM + "/device", "value": did}],
        "deviceName": [{"name": software_name, "type": "user-friendly-name"}],
    }
    if version is not None:
        d["version"] = [{"value": version}]
    return d

def make_imaging_study(sid, pid, when, series_uid):
    return {
        "resourceType": "ImagingStudy",
        "id": sid,
        "status": "available",
        "subject": ref("Patient", pid),
        "started": when,
        "identifier": [{"system": SYSTEM + "/study", "value": sid}],
        "series": [{
            "uid": series_uid,
            "modality": {"system": "http://dicom.nema.org/resources/ontology/DCM", "code": "MR"},
            "instance": []
        }]
    }

def make_observation(oid, pid, sid, did, region, side, value, unit,
                     timepoint, trace_state, trace_rule="TRACE-synth-v1"):
    obs = {
        "resourceType": "Observation",
        "id": oid,
        "status": "final",
        "subject": ref("Patient", pid),
        "focus": [ref("ImagingStudy", sid)],
        "device": ref("Device", did),
        "code": {
            "coding": [{"system": SYSTEM + "/biomarker", "code": f"{region}-volume"}],
            "text": f"{side} {region} volume"
        },
        "bodySite": {
            "coding": [{
                "system": SYSTEM + "/anatomy",
                "code": f"{ANATOMY_CODES[region]}-{side.upper()}",
                "display": f"{side} {region}"
            }]
        },
        "effectiveDateTime": timepoint,
        "valueQuantity": {
            "value": value,
            "unit": unit,
            "system": UCUM,
            "code": unit
        },
        "extension": [
            {
                "url": SYSTEM + "/StructureDefinition/timepoint-label",
                "valueString": "baseline" if "2025-01" in timepoint else "12-month"
            },
            {
                "url": SYSTEM + "/StructureDefinition/trace-state",
                "valueCode": trace_state
            },
            {
                "url": SYSTEM + "/StructureDefinition/trace-rule-version",
                "valueString": trace_rule
            }
        ]
    }
    return obs

def make_provenance(prid, obs_id, study_id, device_id):
    return {
        "resourceType": "Provenance",
        "id": prid,
        "recorded": datetime.now(timezone.utc).isoformat(),
        "target": [ref("Observation", obs_id)],
        "entity": [
            {"role": "source", "what": ref("ImagingStudy", study_id)},
            {"role": "source", "what": ref("Device", device_id)}
        ],
        "agent": [{
            "type": {"text": "synthetic extraction software"},
            "who": ref("Device", device_id)
        }]
    }


In [5]:

# Build 16 clean longitudinal synthetic measurement examples:
# 2 persons × 2 regions × 2 hemispheres × 2 time points = 16 observations.
def build_clean_bundle():
    resources = []
    persons = ["syn-p01", "syn-p02"]

    for pidx, pid in enumerate(persons, start=1):
        did = f"dev-{pid}"
        resources += [make_patient(pid), make_device(did, version="1.0")]

        studies = {
            "baseline": (f"study-{pid}-bl", "2025-01-15T09:00:00Z", f"1.2.840.10008.ICHISYN.{pidx}.1"),
            "12-month": (f"study-{pid}-m12", "2026-01-15T09:00:00Z", f"1.2.840.10008.ICHISYN.{pidx}.2"),
        }

        for label, (sid, dt, uid) in studies.items():
            resources.append(make_imaging_study(sid, pid, dt, uid))

        base_values = {
            ("hippocampus", "left"): 3600 + pidx * 20,
            ("hippocampus", "right"): 3650 + pidx * 20,
            ("entorhinal", "left"): 1750 + pidx * 15,
            ("entorhinal", "right"): 1780 + pidx * 15,
        }

        for region in ["hippocampus", "entorhinal"]:
            for side in ["left", "right"]:
                for label, (sid, dt, uid) in studies.items():
                    suffix = "bl" if label == "baseline" else "m12"
                    oid = f"obs-{pid}-{region}-{side}-{suffix}"
                    value = base_values[(region, side)]
                    if label == "12-month":
                        value = value - (70 if region == "hippocampus" else 35)

                    trace_state = "PASS"
                    resources.append(
                        make_observation(
                            oid, pid, sid, did, region, side,
                            value, "mm3", dt, trace_state
                        )
                    )
                    resources.append(
                        make_provenance(f"prov-{oid}", oid, sid, did)
                    )

    return {
        "resourceType": "Bundle",
        "type": "collection",
        "identifier": {"system": SYSTEM + "/bundle", "value": "phase26-clean"},
        "entry": [{"resource": r} for r in resources]
    }

clean_bundle = build_clean_bundle()

resource_counts = {}
for e in clean_bundle["entry"]:
    rt = e["resource"]["resourceType"]
    resource_counts[rt] = resource_counts.get(rt, 0) + 1

print("Clean bundle resource counts:", resource_counts)
assert resource_counts["Observation"] == 16


Clean bundle resource counts: {'Patient': 2, 'Device': 2, 'ImagingStudy': 4, 'Observation': 16, 'Provenance': 16}


In [6]:

# Project-specific structural contract checker.
# This is NOT an official HL7 FHIR validator.

def index_bundle(bundle):
    idx = {}
    for e in bundle.get("entry", []):
        r = e.get("resource", {})
        if r.get("resourceType") and r.get("id"):
            idx[f"{r['resourceType']}/{r['id']}"] = r
    return idx

def ext_value(obs, suffix, value_key):
    for ext in obs.get("extension", []):
        if ext.get("url", "").endswith(suffix):
            return ext.get(value_key)
    return None

def check_bundle(bundle):
    idx = index_bundle(bundle)
    issues = []
    observations = [r for r in idx.values() if r.get("resourceType") == "Observation"]

    for obs in observations:
        oid = obs["id"]

        # Patient linkage
        subj = obs.get("subject", {}).get("reference")
        if not subj or subj not in idx:
            issues.append((oid, "PATIENT_LINK_LOSS"))

        # Imaging study linkage
        focus_refs = [x.get("reference") for x in obs.get("focus", [])]
        valid_studies = [x for x in focus_refs if x in idx and idx[x].get("resourceType") == "ImagingStudy"]
        if not valid_studies:
            issues.append((oid, "SOURCE_LINK_LOSS"))

        # Device linkage + algorithm version
        dref = obs.get("device", {}).get("reference")
        if not dref or dref not in idx:
            issues.append((oid, "MODEL_PROVENANCE_LOSS"))
        else:
            versions = idx[dref].get("version", [])
            if not versions or not versions[0].get("value"):
                issues.append((oid, "MODEL_PROVENANCE_LOSS"))

        # Value and UCUM unit
        q = obs.get("valueQuantity", {})
        if q.get("value") is None:
            issues.append((oid, "VALUE_MISSING"))
        if not q.get("unit") or q.get("system") != UCUM:
            issues.append((oid, "UNIT_UNRESOLVED"))

        # Anatomy + laterality encoded together
        bcodings = obs.get("bodySite", {}).get("coding", [])
        body_code = bcodings[0].get("code", "") if bcodings else ""
        if not body_code or not (body_code.endswith("-LEFT") or body_code.endswith("-RIGHT")):
            issues.append((oid, "ANATOMY_LATERALITY_LOSS"))

        # TRACE state and rule version
        state = ext_value(obs, "trace-state", "valueCode")
        rule = ext_value(obs, "trace-rule-version", "valueString")
        if state not in TRACE_STATES or not rule:
            issues.append((oid, "TRACE_STATE_LOSS"))

        # Provenance target must point to observation and include study + device.
        provs = [
            r for r in idx.values()
            if r.get("resourceType") == "Provenance"
            and any(t.get("reference") == f"Observation/{oid}" for t in r.get("target", []))
        ]
        if not provs:
            issues.append((oid, "PROVENANCE_CHAIN_LOSS"))
        else:
            entity_refs = {e.get("what", {}).get("reference") for e in provs[0].get("entity", [])}
            if not any(x in entity_refs for x in valid_studies) or dref not in entity_refs:
                issues.append((oid, "PROVENANCE_CHAIN_LOSS"))

    return {
        "observation_count": len(observations),
        "issue_count": len(issues),
        "issues": issues
    }

clean_check = check_bundle(clean_bundle)
print(clean_check)
assert clean_check["observation_count"] == 16
assert clean_check["issue_count"] == 0


{'observation_count': 16, 'issue_count': 0, 'issues': []}



## Controlled synthetic FHIR perturbations

Phase 26 now expresses the Phase 25 challenge families at the FHIR representation layer.

The expected result is not “clinical validity.” The expected result is that the project-specific checker detects the deliberately removed or corrupted evidence dimension.


In [7]:

def clone_bundle():
    return copy.deepcopy(clean_bundle)

def first_obs(bundle):
    return next(
        e["resource"] for e in bundle["entry"]
        if e["resource"].get("resourceType") == "Observation"
    )

def find_resource(bundle, resource_type, rid):
    for e in bundle["entry"]:
        r = e["resource"]
        if r.get("resourceType") == resource_type and r.get("id") == rid:
            return r
    return None

cases = {}

# E0 clean
cases["E0_clean"] = (clone_bundle(), set())

# E1 laterality removed
b = clone_bundle()
o = first_obs(b)
o["bodySite"]["coding"][0]["code"] = "SYN-HIP"
cases["E1_laterality_loss"] = (b, {"ANATOMY_LATERALITY_LOSS"})

# E2 missing unit
b = clone_bundle()
o = first_obs(b)
o["valueQuantity"].pop("unit", None)
o["valueQuantity"].pop("code", None)
cases["E2_missing_unit"] = (b, {"UNIT_UNRESOLVED"})

# E3 broken longitudinal/source linkage
b = clone_bundle()
o = first_obs(b)
o["focus"] = [{"reference": "ImagingStudy/nonexistent-study"}]
cases["E3_source_link_loss"] = (b, {"SOURCE_LINK_LOSS", "PROVENANCE_CHAIN_LOSS"})

# E4 missing model version
b = clone_bundle()
o = first_obs(b)
dref = o["device"]["reference"].split("/", 1)[1]
d = find_resource(b, "Device", dref)
d.pop("version", None)
cases["E4_missing_model_version"] = (b, {"MODEL_PROVENANCE_LOSS"})

# E5 TRACE state stripped
b = clone_bundle()
o = first_obs(b)
o["extension"] = [
    e for e in o.get("extension", [])
    if not e.get("url", "").endswith("trace-state")
]
cases["E5_trace_state_stripped"] = (b, {"TRACE_STATE_LOSS"})

# E6 Provenance removed for the first observation
b = clone_bundle()
o = first_obs(b)
target_ref = f"Observation/{o['id']}"
b["entry"] = [
    e for e in b["entry"]
    if not (
        e["resource"].get("resourceType") == "Provenance"
        and any(t.get("reference") == target_ref for t in e["resource"].get("target", []))
    )
]
cases["E6_provenance_chain_loss"] = (b, {"PROVENANCE_CHAIN_LOSS"})

# E7 combined held-out style failure
b = clone_bundle()
o = first_obs(b)
o["bodySite"]["coding"][0]["code"] = "SYN-HIP"
o["extension"] = [
    e for e in o.get("extension", [])
    if not e.get("url", "").endswith("trace-state")
]
dref = o["device"]["reference"].split("/", 1)[1]
d = find_resource(b, "Device", dref)
d.pop("version", None)
cases["E7_combined_fault"] = (
    b,
    {"ANATOMY_LATERALITY_LOSS", "TRACE_STATE_LOSS", "MODEL_PROVENANCE_LOSS"}
)

rows = []
for case_name, (bundle, expected) in cases.items():
    result = check_bundle(bundle)
    observed = set(code for _, code in result["issues"])
    passed = expected.issubset(observed)
    if case_name == "E0_clean":
        passed = (result["issue_count"] == 0)

    rows.append({
        "fixture": case_name,
        "passed": bool(passed),
        "expected_warning_types": ";".join(sorted(expected)),
        "observed_warning_types": ";".join(sorted(observed)),
        "observation_count": result["observation_count"],
        "issue_count": result["issue_count"],
    })

phase26_summary = pd.DataFrame(rows)
display(phase26_summary)

assert phase26_summary["passed"].all(), "At least one Phase 26 synthetic FHIR contract case failed."
print(f"PASS: {phase26_summary['passed'].sum()}/{len(phase26_summary)} Phase 26 FHIR cases met declared expectations.")


,fixture,passed,expected_warning_types,observed_warning_types,observation_count,issue_count
0,E0_clean,True,,,16,0
1,E1_laterality_loss,True,ANATOMY_LATERALITY_LOSS,ANATOMY_LATERALITY_LOSS,16,1
2,E2_missing_unit,True,UNIT_UNRESOLVED,UNIT_UNRESOLVED,16,1
3,E3_source_link_loss,True,PROVENANCE_CHAIN_LOSS;SOURCE_LINK_LOSS,PROVENANCE_CHAIN_LOSS;SOURCE_LINK_LOSS,16,2
4,E4_missing_model_version,True,MODEL_PROVENANCE_LOSS,MODEL_PROVENANCE_LOSS,16,8
5,E5_trace_state_stripped,True,TRACE_STATE_LOSS,TRACE_STATE_LOSS,16,1
6,E6_provenance_chain_loss,True,PROVENANCE_CHAIN_LOSS,PROVENANCE_CHAIN_LOSS,16,1
7,E7_combined_fault,True,ANATOMY_LATERALITY_LOSS;MODEL_PROVENANCE_LOSS;...,ANATOMY_LATERALITY_LOSS;MODEL_PROVENANCE_LOSS;...,16,10


PASS: 8/8 Phase 26 FHIR cases met declared expectations.


In [8]:

# Reconstruction check on the clean synthetic bundle:
# prove that the required evidence dimensions can be recovered from the FHIR representation.

def reconstruct_evidence_rows(bundle):
    idx = index_bundle(bundle)
    out = []

    for r in idx.values():
        if r.get("resourceType") != "Observation":
            continue

        study_ref = r.get("focus", [{}])[0].get("reference")
        study = idx.get(study_ref, {})
        device_ref = r.get("device", {}).get("reference")
        device = idx.get(device_ref, {})
        body = r.get("bodySite", {}).get("coding", [{}])[0]
        q = r.get("valueQuantity", {})

        out.append({
            "observation_id": r.get("id"),
            "patient_ref": r.get("subject", {}).get("reference"),
            "study_ref": study_ref,
            "acquisition_time": study.get("started"),
            "anatomy_laterality_code": body.get("code"),
            "biomarker": r.get("code", {}).get("text"),
            "value": q.get("value"),
            "unit": q.get("unit"),
            "unit_system": q.get("system"),
            "device_ref": device_ref,
            "algorithm_version": (
                device.get("version", [{}])[0].get("value")
                if device.get("version") else None
            ),
            "trace_state": ext_value(r, "trace-state", "valueCode"),
            "trace_rule_version": ext_value(r, "trace-rule-version", "valueString"),
        })

    return pd.DataFrame(out)

reconstructed = reconstruct_evidence_rows(clean_bundle)
display(reconstructed.head())

required_cols = [
    "patient_ref", "study_ref", "acquisition_time",
    "anatomy_laterality_code", "biomarker",
    "value", "unit", "unit_system",
    "device_ref", "algorithm_version",
    "trace_state", "trace_rule_version"
]

assert len(reconstructed) == 16
assert reconstructed[required_cols].notna().all().all()
assert set(reconstructed["unit_system"]) == {UCUM}
print("PASS: all 16 clean synthetic observations reconstruct every required Phase 26 evidence dimension.")


,observation_id,patient_ref,study_ref,acquisition_time,anatomy_laterality_code,biomarker,value,unit,unit_system,device_ref,algorithm_version,trace_state,trace_rule_version
0,obs-syn-p01-hippocampus-left-bl,Patient/syn-p01,ImagingStudy/study-syn-p01-bl,2025-01-15T09:00:00Z,SYN-HIP-LEFT,left hippocampus volume,3620,mm3,http://unitsofmeasure.org,Device/dev-syn-p01,1.0,PASS,TRACE-synth-v1
1,obs-syn-p01-hippocampus-left-m12,Patient/syn-p01,ImagingStudy/study-syn-p01-m12,2026-01-15T09:00:00Z,SYN-HIP-LEFT,left hippocampus volume,3550,mm3,http://unitsofmeasure.org,Device/dev-syn-p01,1.0,PASS,TRACE-synth-v1
2,obs-syn-p01-hippocampus-right-bl,Patient/syn-p01,ImagingStudy/study-syn-p01-bl,2025-01-15T09:00:00Z,SYN-HIP-RIGHT,right hippocampus volume,3670,mm3,http://unitsofmeasure.org,Device/dev-syn-p01,1.0,PASS,TRACE-synth-v1
3,obs-syn-p01-hippocampus-right-m12,Patient/syn-p01,ImagingStudy/study-syn-p01-m12,2026-01-15T09:00:00Z,SYN-HIP-RIGHT,right hippocampus volume,3600,mm3,http://unitsofmeasure.org,Device/dev-syn-p01,1.0,PASS,TRACE-synth-v1
4,obs-syn-p01-entorhinal-left-bl,Patient/syn-p01,ImagingStudy/study-syn-p01-bl,2025-01-15T09:00:00Z,SYN-ENT-LEFT,left entorhinal volume,1765,mm3,http://unitsofmeasure.org,Device/dev-syn-p01,1.0,PASS,TRACE-synth-v1


PASS: all 16 clean synthetic observations reconstruct every required Phase 26 evidence dimension.


In [9]:

# Save Drive → Drive outputs in the same folder as Phase 25
OUT_JSON = WORKDIR / "ICHI2027_Phase26_SHAREABLE.json"
OUT_CSV = WORKDIR / "ICHI2027_Phase26_FHIR_R4_SYNTHETIC_SUMMARY.csv"
OUT_BUNDLE = WORKDIR / "ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json"

# Public-safe bundle: fully synthetic only.
with open(OUT_BUNDLE, "w", encoding="utf-8") as f:
    json.dump(clean_bundle, f, indent=2)

phase26_summary.to_csv(OUT_CSV, index=False)

shareable = {
    "project": "ICHI2027_neuroimaging_fhir_omop_fidelity",
    "phase": "26_synthetic_fhir_r4_evidence_contract_gate",
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "privacy": "Synthetic-only execution; public-safe outputs; no ADNI records, RIDs, private paths, or real participant identifiers.",
    "phase25_sha256": sha256_file(PHASE25_JSON),
    "phase25_gate_verified": True,
    "phase25_exact_original_generic_C_portable_to_MRI": False,
    "phase25_exact_original_generic_C_MRI_executed": False,
    "fhir_representation": {
        "standard_family": "FHIR R4",
        "resources_used": ["Patient", "ImagingStudy", "Device", "Observation", "Provenance"],
        "formal_HL7_validator_executed": False,
        "project_specific_structural_contract_only": True,
        "clean_synthetic_observation_count": int(clean_check["observation_count"]),
        "clean_synthetic_issue_count": int(clean_check["issue_count"]),
        "reconstructed_required_dimensions_complete": True,
        "synthetic_bundle_sha256": sha256_file(OUT_BUNDLE),
    },
    "controlled_fhir_cases": phase26_summary.to_dict(orient="records"),
    "controlled_fhir_case_count": int(len(phase26_summary)),
    "controlled_fhir_cases_passing_declared_expectations": int(phase26_summary["passed"].sum()),
    "gates": {
        "phase25_verified": True,
        "synthetic_FHIR_R4_representation_created": True,
        "synthetic_required_evidence_reconstructable": True,
        "synthetic_FHIR_contract_cases_passed": bool(phase26_summary["passed"].all()),
        "official_HL7_profile_validation_completed": False,
        "pyOMOP_ETL_executed": False,
        "official_MICDM_load_completed": False,
        "real_MRI_rows_processed": 0,
        "exact_ADNI_units_attested": False,
        "DICOM_UID_semantics_attested": False,
        "Athena_mappings_approved": False,
        "true_four_arm_comparison_completed": False,
        "OMOP_target_cohort_equivalence_tested": False,
    },
    "next_stage_gate": (
        "Freeze and verify the exact OMOP/MI-CDM target revision and source-to-target field contract; "
        "then implement a faithful synthetic Arm A mapping from this FHIR R4 evidence layer. "
        "Do not execute or substitute Arm C, and do not claim real-data or official MI-CDM validation."
    )
}

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(shareable, f, indent=2)

print("Saved:")
print(" -", OUT_JSON)
print(" -", OUT_CSV)
print(" -", OUT_BUNDLE)


Saved:
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SHAREABLE.json
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_FHIR_R4_SYNTHETIC_SUMMARY.csv
 - /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json


In [10]:

# Final read-back verification
with open(OUT_JSON, "r", encoding="utf-8") as f:
    verify_json = json.load(f)

verify_csv = pd.read_csv(OUT_CSV)

assert verify_json["gates"]["phase25_verified"] is True
assert verify_json["gates"]["synthetic_FHIR_R4_representation_created"] is True
assert verify_json["gates"]["synthetic_required_evidence_reconstructable"] is True
assert verify_json["gates"]["synthetic_FHIR_contract_cases_passed"] is True
assert verify_json["gates"]["pyOMOP_ETL_executed"] is False
assert verify_json["gates"]["official_MICDM_load_completed"] is False
assert verify_json["gates"]["real_MRI_rows_processed"] == 0
assert verify_csv["passed"].all()

print("PHASE 26 COMPLETE")
print("All saved outputs were read back successfully from Google Drive.")
print("Next phase: exact pinned OMOP/MI-CDM target contract + faithful synthetic Arm A mapping.")


PHASE 26 COMPLETE
All saved outputs were read back successfully from Google Drive.
Next phase: exact pinned OMOP/MI-CDM target contract + faithful synthetic Arm A mapping.
